# FracID Real-Data Case Study: 1978 English Boarding School Influenza Outbreak

This notebook reproduces the real-world dynamical system identification case study for **FracID**:
- **Dataset**: Daily time-series of an Influenza A (H1N1) outbreak in a boys' boarding school (North of England, 1978).
- **Source**: *British Medical Journal*, 1978; 1(6112): 587–588.
- **Population**:  = 763$ students (confined, closed community).
- **Model**: Caputo Fractional-Order SIR Epidemic Model with Memory (zsh < lpha \le 1$):
  2858egin{aligned}
  {}^C D^lpha S &= -eta rac{S I}{N} \
  {}^C D^lpha I &= eta rac{S I}{N} - \gamma I \
  {}^C D^lpha R &= \gamma I
  \end{aligned}2858
  where $eta$ is the transmission rate, $\gamma$ is the recovery rate, and  = eta / \gamma$.
- **Objectives**:
  1. Load the real epidemiological outbreak data.
  2. Formulate the bound-constrained non-linear least squares problem.
  3. Estimate fractional order $lpha$ and biological parameters $ via batched Differential Evolution.
  4. Compare the fractional model against the classical integer-order (Markovian) model ($lpha = 1$).
  5. Quantify parameter uncertainty using residual bootstrap and profile-likelihood confidence intervals.
  6. Visualize fitted trajectories against recorded hospital/bed-ridden case counts.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fracid.models import FractionalSIR
from fracid.data import Dataset
from fracid.estimation import FitProblem, fit_least_squares
from fracid.diagnostics import (
    compare_fractional_vs_integer,
    compute_metrics,
    run_bootstrap,
    compute_alpha_sensitivity,
    compute_profile_ci,
    plot_trajectory_matplotlib,
    plot_residuals_matplotlib
)

print("FracID modules successfully imported.")


## 1. Load and Inspect Real Outbreak Observations

In [ ]:
data_path = os.path.join("..", "data", "boarding_school_influenza_1978.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "boarding_school_influenza_1978.csv")

df = pd.read_csv(data_path)
print(f"Loaded {len(df)} daily observations:")
display(df)

# Plot raw epidemic curve
fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.plot(df["t"], df["I"], "ro-", lw=2, ms=6, label="Infectious I(t) [Bed-ridden]")
ax.plot(df["t"], df["R"], "bs--", lw=1.5, ms=5, label="Convalescent R(t)")
ax.plot(df["t"], df["S"], "k:^", lw=1.5, ms=5, label="Susceptible S(t)")
ax.set_xlabel("Time (days)", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of Boys", fontsize=11, fontweight="bold")
ax.set_title("1978 English Boarding School Influenza Outbreak Data", fontsize=12, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(loc="center right")
plt.tight_layout()
plt.show()


## 2. Set Up Fractional SIR Identification Problem

In [ ]:
dataset = Dataset.from_frame(
    df,
    time_col="t",
    value_cols=["S", "I", "R"],
    meta=dict(model="Fractional SIR", N=763.0, source="BMJ 1978")
)

# Total population N=763 is fixed; initial conditions from day 0: S(0)=760, I(0)=3, R(0)=0
model = FractionalSIR(params=dict(beta=1.6, gamma=0.45, N=763.0), x0=[760.0, 3.0, 0.0])

problem = FitProblem(
    model=model,
    data=dataset,
    free=["beta", "gamma"],
    alpha_bounds=(0.6, 1.0),
    bounds=dict(beta=(0.2, 3.5), gamma=(0.05, 1.5)),
    x0_mode="data"
)

print(f"FitProblem initialized with {problem.n_par} parameters to estimate: {problem.names}")


## 3. Fit Fractional-Order Model via Differential Evolution

In [ ]:
frac_res = fit_least_squares(
    problem,
    method="differential-evolution",
    max_iter=80,
    popsize=12,
    polish_iter=80,
    seed=42
)

print(f"=== Fractional Model Identification Outcome ===")
print(f"Estimated fractional order alpha: {frac_res.alpha:.4f}")
print(f"Estimated beta:                   {frac_res.params['beta']:.4f}")
print(f"Estimated gamma:                  {frac_res.params['gamma']:.4f}")
print(f"Implied basic reproduction R0:    {frac_res.params['beta'] / frac_res.params['gamma']:.2f}")
print(f"Residual RMSE (students):         {frac_res.rmse:.2f}")
print(f"Runtime:                          {frac_res.runtime:.2f} s")


## 4. Compare with Standard Integer-Order (Markovian) Model (alpha = 1)

In [ ]:
int_res = fit_least_squares(
    problem.integer_order(),
    method="differential-evolution",
    max_iter=80,
    popsize=12,
    polish_iter=80,
    seed=42
)

print(f"Integer-order (alpha = 1.0) fit completed in {int_res.runtime:.2f} s")

comp_df = compare_fractional_vs_integer(frac_res, int_res)
display(comp_df)


## 5. Trajectory Fits and Residual Diagnostics

In [ ]:
fig_tr = plot_trajectory_matplotlib(frac_res, dpi=120)
plt.show()

fig_res = plot_residuals_matplotlib(frac_res, kind="time")
plt.show()


## 6. Uncertainty Quantification: Residual Bootstrap and Profile Likelihood CIs

In [ ]:
print("Running 30 residual bootstrap replicates...")
boot_res = run_bootstrap(frac_res, n_boot=30, seed=42)
display(boot_res.summary)

print("Computing Profile Likelihood curve J(alpha)...")
sens_df = compute_alpha_sensitivity(frac_res, n_points=21, mode="slice")
ci_lo, ci_hi, threshold = compute_profile_ci(frac_res, sens_df, confidence=0.95)

print(f"Profile-likelihood 95% CI for alpha: [{ci_lo:.4f}, {ci_hi:.4f}] (threshold J = {threshold:.4g})")
